# Объединение и очистка данных

## Описание
Объединяет три источника в единый датасет:
1. `data/vacancies_trudvsem.csv` (~3 000, полный текст).
2. `data/hh_it_vacancies.csv` (6 968, структура).
3. `data/hh_vacancies_2026_07.csv` (129 464, фильтруем до IT → ~5 850).

## Что делает
1. Фильтрует hh 2026 по `category == "Информационные технологии"`.
2. Унифицирует `experience` (4 категории) и `area_name` (Москва/СПб/регионы).
3. Приводит зарплаты к «месяц, net (на руки)».
4. Объединяет с `source_type`.
5. Фильтрует выбросы (`salary_net` в [15k, 600k]).
6. Фильтрует старые вакансии (`published_at >= 2024-01-01`).
7. Сохраняет `data/combined.csv`.

## Результат
~7 261 IT-вакансия с зарплатой, 3 источника.

In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 250)

df_trud = pd.read_csv("data/vacancies_trudvsem.csv")
df_hh_it = pd.read_csv("data/hh_it_vacancies.csv")
df_hh_2026 = pd.read_csv("data/hh_vacancies_2026_07.csv")

print(f"trudvsem: {df_trud.shape}")
print(f"hh IT 2026: {df_hh_it.shape}")
print(f"hh 2026: {df_hh_2026.shape}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print(f"hh 2026 до фильтра: {len(df_hh_2026)}")
df_hh_2026 = df_hh_2026[df_hh_2026["category"] == "Информационные технологии"].copy()
print(f"hh 2026 IT: {len(df_hh_2026)}")

In [ ]:
def normalize_experience(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower().strip()
    if x in ("без опыта", "noexperience", "no experience"):
        return "no_experience"
    if x in ("1-3 года", "between1and3"):
        return "1_3_years"
    if x in ("3-6 лет", "between3and6"):
        return "3_6_years"
    if x in ("более 6 лет", "morethan6"):
        return "6_plus_years"
    return np.nan

def normalize_region(x):
    if pd.isna(x):
        return "unknown"
    x = str(x).lower().strip()
    x = re.sub(r"^(город|г\.|гор\.)\s*", "", x)
    x = x.replace("  ", " ").strip()
    if x in ("москва", "moscow"):
        return "Москва"
    if x in ("санкт-петербург", "спб", "saint-petersburg"):
        return "Санкт-Петербург"
    return x.title()

In [ ]:
df_trud_clean = pd.DataFrame()
df_trud_clean["vacancy_id"] = df_trud["id"].astype(str)
df_trud_clean["title"] = df_trud["title"]
df_trud_clean["description"] = df_trud["description"]
df_trud_clean["skills"] = df_trud["skills"]
df_trud_clean["qualification"] = df_trud["qualification"]
df_trud_clean["company_name"] = df_trud["company_name"]
df_trud_clean["area_name"] = df_trud["region_name"].apply(normalize_region)
df_trud_clean["experience"] = np.nan
df_trud_clean["published_at"] = pd.to_datetime(df_trud["date_create"], errors="coerce")
df_trud_clean["salary_from"] = pd.to_numeric(df_trud["salary_min"], errors="coerce")
df_trud_clean["salary_to"] = pd.to_numeric(df_trud["salary_max"], errors="coerce")
df_trud_clean["gross"] = np.nan
df_trud_clean["work_formats"] = df_trud["schedule"]
df_trud_clean["employer_rating"] = np.nan
df_trud_clean["responses_count"] = np.nan
df_trud_clean["professional_role_id"] = np.nan
df_trud_clean["source_type"] = "trudvsem"

print(f"trudvsem clean: {df_trud_clean.shape}")

In [ ]:
df_hh_it_clean = pd.DataFrame()
df_hh_it_clean["vacancy_id"] = df_hh_it["vacancy_id"].astype(str)
df_hh_it_clean["title"] = df_hh_it["name"]
df_hh_it_clean["description"] = np.nan
df_hh_it_clean["skills"] = np.nan
df_hh_it_clean["qualification"] = np.nan
df_hh_it_clean["company_name"] = df_hh_it["employer_name"]
df_hh_it_clean["area_name"] = df_hh_it["city"].apply(normalize_region)
df_hh_it_clean["experience"] = df_hh_it["experience"].apply(normalize_experience)
df_hh_it_clean["published_at"] = pd.to_datetime(df_hh_it["published_at"], errors="coerce")
df_hh_it_clean["salary_from"] = pd.to_numeric(df_hh_it["salary_from"], errors="coerce")
df_hh_it_clean["salary_to"] = pd.to_numeric(df_hh_it["salary_to"], errors="coerce")
df_hh_it_clean["gross"] = df_hh_it["salary_gross"]
df_hh_it_clean["work_formats"] = df_hh_it["work_formats"]
df_hh_it_clean["employer_rating"] = df_hh_it["employer_rating"]
df_hh_it_clean["responses_count"] = df_hh_it["responses_count"]
df_hh_it_clean["professional_role_id"] = df_hh_it["professional_role_id"]
df_hh_it_clean["source_type"] = "hh_it_2026"

print(f"hh IT clean: {df_hh_it_clean.shape}")

In [ ]:
df_hh_2026_clean = pd.DataFrame()
df_hh_2026_clean["vacancy_id"] = df_hh_2026["vacancy_id"].astype(str)
df_hh_2026_clean["title"] = df_hh_2026["name"]
df_hh_2026_clean["description"] = np.nan
df_hh_2026_clean["skills"] = np.nan
df_hh_2026_clean["qualification"] = np.nan
df_hh_2026_clean["company_name"] = df_hh_2026["company"]
df_hh_2026_clean["area_name"] = df_hh_2026["area_name"].apply(normalize_region)
df_hh_2026_clean["experience"] = df_hh_2026["experience"].apply(normalize_experience)
df_hh_2026_clean["published_at"] = pd.to_datetime(df_hh_2026["publication_time"], errors="coerce")
df_hh_2026_clean["salary_from"] = pd.to_numeric(df_hh_2026["salary_from_rub"], errors="coerce")
df_hh_2026_clean["salary_to"] = pd.to_numeric(df_hh_2026["salary_to_rub"], errors="coerce")
df_hh_2026_clean["gross"] = df_hh_2026["gross"]
df_hh_2026_clean["work_formats"] = df_hh_2026["work_formats"]
df_hh_2026_clean["employer_rating"] = np.nan
df_hh_2026_clean["responses_count"] = df_hh_2026["responses_count"]
df_hh_2026_clean["professional_role_id"] = df_hh_2026["role_ids"]
df_hh_2026_clean["source_type"] = "hh_2026"

print(f"hh 2026 clean: {df_hh_2026_clean.shape}")

In [ ]:
df_all = pd.concat([df_trud_clean, df_hh_it_clean, df_hh_2026_clean], ignore_index=True, sort=False)
print(f"Объединено: {len(df_all)}")
print(df_all["source_type"].value_counts())

In [ ]:
df_all["salary_avg"] = df_all[["salary_from", "salary_to"]].mean(axis=1)
df_all["tax_mult"] = np.where(df_all["gross"] == True, 0.87, 1.0)
df_all["salary_net"] = df_all["salary_avg"] * df_all["tax_mult"]

before = len(df_all)
df_all = df_all[(df_all["salary_net"] >= 15_000) & (df_all["salary_net"] <= 600_000)].copy()
print(f"После фильтра зарплат: {len(df_all)} (отброшено {before - len(df_all)})")

In [ ]:
# Парсим даты (важно: строки сравниваются лексикографически, поэтому явный to_datetime)
df_all["published_at"] = pd.to_datetime(
    df_all["published_at"].astype(str).str.strip(),
    errors="coerce",
    format="mixed",
    utc=True,
)

before = len(df_all)
df_all = df_all[df_all["published_at"] >= "2024-01-01"].copy()
print(f"После фильтра 2024+: {len(df_all)} (отброшено {before - len(df_all)} старых)")

In [ ]:
print("\n" + "="*60)
print("ИТОГОВАЯ СТАТИСТИКА")
print("="*60)
print(f"\nsalary_net по источникам:")
print(df_all.groupby("source_type")["salary_net"].agg(["count", "median"]).round(0))

print(f"\nexperience:")
print(df_all["experience"].value_counts(dropna=False))

print(f"\nС текстом:")
print(df_all["description"].notna().sum())

print(f"\nТоп-15 городов:")
print(df_all["area_name"].value_counts().head(15))

# Сохранение
df_all.to_csv("data/combined.csv", index=False, encoding="utf-8-sig")
print(f"\n✓ Сохранено: data/combined.csv ({len(df_all)} строк)")